In [4]:
############################################################
# 1. 데이터 전처리 (Preprocessing)
############################################################

# --- 1-1. 기본 라이브러리 임포트 ---
import pandas as pd  
import json          
import os            
import re            
import warnings     

# --- 1-2. 설정 ---
warnings.filterwarnings('ignore')

# --- 1-3. 경로 설정 ---
# 현재 .ipynb 파일이 있는 'notebooks' 폴더 기준으로 상위 폴더(.)가 기준
BASE_DIR = '../'
# 원본 데이터가 있는 'data' 폴더 경로
DATA_DIR = os.path.join(BASE_DIR, 'data')
# 전처리/임베딩 결과물을 저장할 'output' 폴더 경로
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

# 만약 'output' 폴더가 존재하지 않는다면,
# exist_ok=True 옵션으로 오류 없이 폴더를 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 설정된 경로 확인
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# --- 1-4. 원본 파일 경로 정의 ---
# 4개의 원본 파일 경로를 딕셔너리 형태로 미리 정의
review_files = {
    'instruments': os.path.join(DATA_DIR, 'Musical_Instruments.jsonl'),
    'music': os.path.join(DATA_DIR, 'CDs_and_Vinyl.jsonl')
}

meta_files = {
    'instruments': os.path.join(DATA_DIR, 'meta_Musical_Instruments.jsonl'),
    'music': os.path.join(DATA_DIR, 'meta_CDs_and_Vinyl.jsonl')
}

Data directory: ../data
Output directory: ../output


In [5]:
# .jsonl 파일은 한 줄이 하나의 JSON 객체입니다.
# 대용량 파일을 고려하여, 파일을 통째로 읽지 않고 한 줄씩 읽어오는 함수를 정의합니다.

def load_jsonl_to_df(file_path):
    """
    .jsonl 파일을 한 줄씩 읽어 Pandas DataFrame으로 로드합니다.
    """
    # 데이터를 담을 빈 리스트 초기화
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # 파일을 한 줄(line)씩 순회
            for line in f:
                try:
                    # 각 줄(line)은 문자열이므로 json.loads를 사용해 Python 딕셔너리로 변환
                    # 변환된 딕셔너리를 data 리스트에 추가
                    data.append(json.loads(line))
                except json.JSONDecodeError:
                    # 간혹 파일에 손상된 줄이 있을 경우, 오류를 출력하고 무시
                    print(f"Skipping a bad line in {file_path}")
    except FileNotFoundError:
        # 만약 파일 경로가 잘못되었다면 오류 메시지 출력
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame()
        
    # 모든 데이터를 담은 리스트(data)를 pd.DataFrame으로 변환하여 반환
    return pd.DataFrame(data)

print("Helper function 'load_jsonl_to_df' defined.")

Helper function 'load_jsonl_to_df' defined.


In [6]:
# Cell 2에서 정의한 함수를 사용해 4개의 파일을 모두 DataFrame으로 불러옵니다.

# 1. 리뷰 데이터 로드
inst_reviews_df = load_jsonl_to_df(review_files['instruments'])
cds_reviews_df = load_jsonl_to_df(review_files['music'])

# 2. 메타 데이터 로드
inst_meta_df = load_jsonl_to_df(meta_files['instruments'])
cds_meta_df = load_jsonl_to_df(meta_files['music'])

# 3. 각 DataFrame의 정보(컬럼명, 타입, 결측치)와 상위 5개 행(head)을 출력하여 확인
print("--- Review Data (Instruments) ---")
inst_reviews_df.info()
print(inst_reviews_df.head())

print("\n--- Meta Data (Instruments) ---")
inst_meta_df.info()
print(inst_meta_df.head())

print("\n--- Review Data (CDs & Vinyl) ---")
cds_reviews_df.info()
print(cds_reviews_df.head()) # (이때 'user_id', 'rating' 컬럼 확인)

print("\n--- Meta Data (CDs & Vinyl) ---")
cds_meta_df.info()
print(cds_meta_df.head()) # (이때 'parent_asin' 컬럼 확인)

--- Review Data (Instruments) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3017439 entries, 0 to 3017438
Data columns (total 10 columns):
 #   Column             Dtype  
---  ------             -----  
 0   rating             float64
 1   title              object 
 2   text               object 
 3   images             object 
 4   asin               object 
 5   parent_asin        object 
 6   user_id            object 
 7   timestamp          int64  
 8   helpful_vote       int64  
 9   verified_purchase  bool   
dtypes: bool(1), float64(1), int64(2), object(6)
memory usage: 210.1+ MB
   rating                                             title  \
0     5.0                                        Five Stars   
1     3.0  nice sound.  pedal failed after less than 1 year   
2     4.0                                              okay   
3     3.0                                   Easy to return.   
4     5.0                  Good product despite tight bolt.   

                 

In [7]:
# [문제] 'CDs_and_Vinyl' 리뷰는 'user_id', 'rating'을 사용하고,
#       'Musical_Instruments' 리뷰는 'reviewerID', 'overall'을 사용합니다.
#  협업 필터링을 할 수 있도록 컬럼명을 하나로 통일합니다.

print(f"CDs & Vinyl (Before): {cds_reviews_df.columns.to_list()}")

# .rename() 함수를 사용해 'CDs_and_Vinyl'의 컬럼명을 변경
cds_reviews_df = cds_reviews_df.rename(columns={
    'user_id': 'reviewerID',  # user_id -> reviewerID (사용자 ID)
    'rating': 'overall',      # rating -> overall (평점)
    'review': 'reviewText'    # review -> reviewText (리뷰 텍스트)
})

print(f"CDs & Vinyl (After): {cds_reviews_df.columns.to_list()}")
print("컬럼명 표준화 완료.")

CDs & Vinyl (Before): ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
CDs & Vinyl (After): ['overall', 'title', 'text', 'images', 'asin', 'parent_asin', 'reviewerID', 'timestamp', 'helpful_vote', 'verified_purchase']
컬럼명 표준화 완료.


In [8]:
# 4개의 DataFrame을 2개(리뷰, 메타)로 합칩니다.
# [중요] 합치기 전에, 'domain' 컬럼을 추가하여 이 아이템이 'instruments'인지 'music'인지 구분합니다.

# 1. 리뷰 데이터 병합
inst_reviews_df['domain'] = 'instruments' # 악기 리뷰에 'instruments' 태그 추가
cds_reviews_df['domain'] = 'music'       # 음반 리뷰에 'music' 태그 추가
# pd.concat으로 두 DataFrame을 위아래로 합침. ignore_index=True로 인덱스 재설정
all_reviews_df = pd.concat([inst_reviews_df, cds_reviews_df], ignore_index=True)

# 2. 메타 데이터 병합
inst_meta_df['domain'] = 'instruments'
cds_meta_df['domain'] = 'music'
all_meta_df = pd.concat([inst_meta_df, cds_meta_df], ignore_index=True)

# 3. (선택적) 메모리 관리를 위해 원본 DataFrame 변수 삭제
del inst_reviews_df, cds_reviews_df, inst_meta_df, cds_meta_df

# 4. 통합 결과 확인
print("--- 통합된 리뷰 데이터 (all_reviews_df) ---")
print(f"Total reviews: {len(all_reviews_df)}")
print(all_reviews_df['domain'].value_counts()) # 도메인별 리뷰 수 확인

print("\n--- 통합된 메타 데이터 (all_meta_df) ---")
print(f"Total items (meta): {len(all_meta_df)}")
print(all_meta_df['domain'].value_counts()) # 도메인별 아이템 수 확인

--- 통합된 리뷰 데이터 (all_reviews_df) ---
Total reviews: 7844712
domain
music          4827273
instruments    3017439
Name: count, dtype: int64

--- 통합된 메타 데이터 (all_meta_df) ---
Total items (meta): 915552
domain
music          701959
instruments    213593
Name: count, dtype: int64


In [9]:
# [문제] Cell 3에서 확인했듯이, 리뷰 데이터(all_reviews_df)는 'asin'을 아이템 ID로 사용하지만,
#       메타 데이터(all_meta_df)는 'parent_asin'을 아이템 ID로 사용합니다.
# [해결] 두 데이터를 연결하는 "공통 열쇠"가 필요하므로, 메타 데이터의 'parent_asin'을 'asin'으로 변경합니다.

print("--- 컬럼명 변경 전 (메타) ---")
print(all_meta_df.columns.to_list())

# 'parent_asin' 컬럼이 메타 데이터에 존재하는지 확인
if 'parent_asin' in all_meta_df.columns:
    # 존재한다면, .rename()을 사용해 'parent_asin' -> 'asin'으로 변경
    all_meta_df = all_meta_df.rename(columns={'parent_asin': 'asin'})
    print("\n[Success] 'parent_asin' -> 'asin'으로 변경 완료.")
else:
    print("\n[Info] 'parent_asin' 컬럼이 없거나 이미 'asin' 컬럼이 존재합니다.")

# 4. 최종 확인
print("\n--- 컬럼명 변경 후 (메타) ---")
print(all_meta_df.columns.to_list()) # 'asin' 컬럼이 생성되었는지 확인

--- 컬럼명 변경 전 (메타) ---
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author', 'domain']

[Success] 'parent_asin' -> 'asin'으로 변경 완료.

--- 컬럼명 변경 후 (메타) ---
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'asin', 'bought_together', 'subtitle', 'author', 'domain']


In [10]:
# 협업 필터링(CF)에 필수적인 컬럼: reviewerID, asin, overall
review_essential_cols = ['reviewerID', 'asin', 'overall']

print(f"리뷰 데이터 결측치 처리 전: {len(all_reviews_df)} 행")
# .isnull().sum()으로 필수 컬럼들의 결측치 개수 확인
print(all_reviews_df[review_essential_cols].isnull().sum())

# .dropna()로 필수 컬럼(subset) 중 하나라도 NaN인 행(row)을 제거
all_reviews_df = all_reviews_df.dropna(subset=review_essential_cols)

# 'overall' 컬럼(평점)이 혹시 문자열일 수 있으므로, pd.to_numeric으로 숫자형 변환
# errors='coerce' 옵션은 변환 실패 시 해당 값을 NaN으로 만듦
all_reviews_df['overall'] = pd.to_numeric(all_reviews_df['overall'], errors='coerce')

# 숫자 변환 중 NaN이 생겼을 수 있으므로, .dropna()로 'overall' 결측치를 다시 한번 제거
all_reviews_df = all_reviews_df.dropna(subset=['overall'])

# 평점(overall) 타입을 float (실수)으로 명확히 지정
all_reviews_df['overall'] = all_reviews_df['overall'].astype(float) 

print(f"리뷰 데이터 결측치 처리 후: {len(all_reviews_df)} 행")

리뷰 데이터 결측치 처리 전: 7844712 행
reviewerID    3017439
asin                0
overall       3017439
dtype: int64
리뷰 데이터 결측치 처리 후: 4827273 행


In [11]:
# 역할의 콘텐츠 기반(CB)에 필수적인 컬럼: asin, title, description
# (Cell 5.5에서 'parent_asin'을 'asin'으로 바꿨기 때문에 이 코드가 정상 작동)
meta_essential_cols = ['asin', 'title', 'description']

print(f"메타 데이터 결측치 처리 전: {len(all_meta_df)} 행")
# 필수 컬럼들의 결측치 개수 확인
print(all_meta_df[meta_essential_cols].isnull().sum())

# .dropna()로 CB 필수 컬럼(subset) 중 하나라도 NaN인 행을 제거
all_meta_df = all_meta_df.dropna(subset=meta_essential_cols)

print(f"메타 데이터 결측치 처리 후: {len(all_meta_df)} 행")

메타 데이터 결측치 처리 전: 915552 행
asin           0
title          0
description    0
dtype: int64
메타 데이터 결측치 처리 후: 915552 행


In [12]:
# [문제] 데이터가 수백만 건으로 너무 크고(sparse), 리뷰가 1~2개뿐인 유저/아이템(Cold Start)은
#       모델 학습에 방해가 됩니다.
# [해결] 최소 N개/M개 리뷰 기준을 정해, '활성' 유저와 '인기' 아이템만 남깁니다.

# 기준 설정 (이 값은 EDA 후 조정 가능)
N_REVIEWS_ITEM = 10  # 아이템은 최소 10개 이상의 리뷰를 받아야 함 --> 조정가능
M_REVIEWS_USER = 5   # 사용자는 최소 5회 이상의 리뷰를 남겨야 함 --> 조정가능

print(f"--- 데이터 축소 시작 (Item >= {N_REVIEWS_ITEM}, User >= {M_REVIEWS_USER}) ---")
print(f"원본 리뷰 수: {len(all_reviews_df)}")

# --- 1. 아이템 필터링 (N=10) ---
# 'asin'별로 리뷰 개수를 셈 (.value_counts())
item_review_counts = all_reviews_df['asin'].value_counts()
# 리뷰 개수가 N개 이상인 아이템('asin') 리스트를 'hot_items'로 저장
hot_items = item_review_counts[item_review_counts >= N_REVIEWS_ITEM].index
# 'hot_items' 리스트에 포함된 'asin'을 가진 리뷰만 남김 (.isin() 사용)
reviews_filtered_items = all_reviews_df[all_reviews_df['asin'].isin(hot_items)]

print(f"N (Item) 필터링 후 리뷰 수: {len(reviews_filtered_items)}")

# --- 2. 사용자 필터링 (M=5) ---
# (N 필터링된 데이터에서) 'reviewerID'별로 리뷰 개수를 셈
user_review_counts = reviews_filtered_items['reviewerID'].value_counts()
# 리뷰 개수가 M개 이상인 사용자('reviewerID') 리스트를 'hot_users'로 저장
hot_users = user_review_counts[user_review_counts >= M_REVIEWS_USER].index
# 'hot_users' 리스트에 포함된 'reviewerID'를 가진 리뷰만 남김
reviews_final = reviews_filtered_items[reviews_filtered_items['reviewerID'].isin(hot_users)]

print(f"M (User) 필터링 후 리뷰 수: {len(reviews_final)}")

# --- 3. 메타데이터 동기화 (중요) ---
# 최종 리뷰(reviews_final)에 살아남은 'asin' 리스트 추출
final_asins = reviews_final['asin'].unique()
# 메타 데이터(all_meta_df)도 이 'final_asins'에 포함된 아이템만 남김
meta_final = all_meta_df[all_meta_df['asin'].isin(final_asins)]

# --- 4. 리뷰데이터 최종 동기화 (확인 사살) ---
# (메타 데이터에 없는 asin이 리뷰에 남았을 경우를 대비해 한번 더 필터링)
final_meta_asins = meta_final['asin'].unique()
reviews_final = reviews_final[reviews_final['asin'].isin(final_meta_asins)]


print("--- 데이터 축소 완료 ---")
print(f"최종 리뷰 수: {len(reviews_final)}")
print(f"최종 사용자 수: {reviews_final['reviewerID'].nunique()}")
print(f"최종 아이템 수: {reviews_final['asin'].nunique()} (리뷰 기준)")
print(f"최종 메타데이터 수: {len(meta_final)} (메타 기준)") # (아이템 수와 메타 수 일치 확인)

# 최종 데이터프레임으로 변수명 덮어쓰기
all_reviews_df = reviews_final
all_meta_df = meta_final

--- 데이터 축소 시작 (Item >= 10, User >= 5) ---
원본 리뷰 수: 4827273
N (Item) 필터링 후 리뷰 수: 3406725
M (User) 필터링 후 리뷰 수: 1521598
--- 데이터 축소 완료 ---
최종 리뷰 수: 1521519
최종 사용자 수: 124791
최종 아이템 수: 86889 (리뷰 기준)
최종 메타데이터 수: 86889 (메타 기준)


In [13]:
# TF-IDF에 사용할 텍스트 데이터를 만듭니다.
# 'title'과 'description' 컬럼을 하나로 합쳐 'content'라는 새 컬럼을 만듭니다.

# (안전장치) description이 간혹 ['desc1', 'desc2'] 처럼 리스트일 수 있습니다.
# 이를 대비해, 리스트면 공백으로 join하고, 아니면 그냥 str(문자열)로 변환하는 함수
def safe_join(field):
    if isinstance(field, list):
        return ' '.join(field)
    return str(field)

# .apply()를 사용해 'description'과 'title'을 안전하게 문자열로 변환
all_meta_df['description_str'] = all_meta_df['description'].apply(safe_join)
all_meta_df['title_str'] = all_meta_df['title'].apply(safe_join)

# 'content' 컬럼 생성: 'title' 문자열과 'description' 문자열을 공백으로 연결
all_meta_df['content'] = all_meta_df['title_str'] + ' ' + all_meta_df['description_str']

print("Content 컬럼 생성 완료.")
print(all_meta_df[['title_str', 'description_str', 'content']].head())

Content 컬럼 생성 완료.
                                         title_str  \
213593                        Release Some Tension   
213607                                  Elvis Live   
213610  Jekyll & Hyde 2012 Concept Recording / Var   
213631  Celebration of Gospel - Taking You Higher!   
213633        Yellowstone for Violin and Orchestra   

                                          description_str  \
213593                         Swv ~ Release Some Tension   
213607                         Presley,Elvis ~ Elvis Live   
213610  2012 concept recording of the classic musical ...   
213631  Product Description BET takes you higher with ...   
213633  Product Description Inspired by countless ride...   

                                                  content  
213593    Release Some Tension Swv ~ Release Some Tension  
213607              Elvis Live Presley,Elvis ~ Elvis Live  
213610  Jekyll & Hyde 2012 Concept Recording / Var 201...  
213631  Celebration of Gospel - Taking You Higher!

In [14]:
# TF-IDF가 단어의 의미를 정확히 파악하도록, 'content'에서 불필요한 문자들을 제거합니다.

def clean_text(text):
    """
    텍스트 정제 함수 (TF-IDF 입력용):
    1. 소문자 변환 (e.g., Guitar -> guitar)
    2. HTML 태그 제거 (e.g., <br> -> )
    3. 알파벳(a-z)과 공백(\s)을 제외한 모든 문자 제거 (숫자, 특수문자)
    4. 다중 공백(  ) -> 단일 공백( )으로 변경
    """
    # 1. 소문자 변환
    text = str(text).lower()
    # 2. HTML 태그 제거 (정규표현식: <로 시작하고 >로 끝나는 모든 패턴)
    text = re.sub(r'<[^>]+>', '', text)
    # 3. 알파벳, 공백 제외 모두 제거 (정규표현식: ^는 '부정'. a-z와 \s가 아닌 모든 것)
    text = re.sub(r'[^a-z\s]', '', text)
    # 4. 다중 공백 처리 (정규표현식: \s+는 1개 이상의 공백)
    text = re.sub(r'\s+', ' ', text).strip() # .strip()은 양끝 공백 제거
    
    return text

print("텍스트 정제 함수 정의 완료. 'content' 컬럼에 적용 중...")

# .apply()를 사용해 'content' 컬럼의 모든 텍스트에 clean_text 함수 적용
# 그 결과를 'cleaned_content'라는 새 컬럼에 저장
all_meta_df['cleaned_content'] = all_meta_df['content'].apply(clean_text)

print("텍스트 정제 완료.")
# 원본 'content'와 정제된 'cleaned_content' 비교 출력
print(all_meta_df[['content', 'cleaned_content']].head())

텍스트 정제 함수 정의 완료. 'content' 컬럼에 적용 중...
텍스트 정제 완료.
                                                  content  \
213593    Release Some Tension Swv ~ Release Some Tension   
213607              Elvis Live Presley,Elvis ~ Elvis Live   
213610  Jekyll & Hyde 2012 Concept Recording / Var 201...   
213631  Celebration of Gospel - Taking You Higher! Pro...   
213633  Yellowstone for Violin and Orchestra Product D...   

                                          cleaned_content  
213593      release some tension swv release some tension  
213607                 elvis live presleyelvis elvis live  
213610  jekyll hyde concept recording var concept reco...  
213631  celebration of gospel taking you higher produc...  
213633  yellowstone for violin and orchestra product d...  


In [15]:
# 전처리 완료
# 정제된 데이터를 저장합니다.

# --- 1. 리뷰 데이터 저장 (CSV) ---
# 유정님이 CF 모델에, 선우님이 시각화에 사용
reviews_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_reviews.csv')
# .to_csv()로 저장. index=False는 DataFrame의 인덱스(0,1,2...)를 저장하지 않음
all_reviews_df.to_csv(reviews_output_path, index=False, encoding='utf-8')
print(f"리뷰 데이터 저장 완료: {reviews_output_path} ({len(all_reviews_df)} 행)")

# --- 2. 메타 데이터 저장 (Pickle) ---
# [Why Pickle?] 메타 데이터의 'categories' 같은 컬럼은 [리스트] 형태입니다.
# CSV로 저장하면 이 [리스트]가 깨져서 "문자열"이 됩니다.
# Pickle(.pkl)은 DataFrame의 구조(리스트, 타입)를 그대로 보존하며 저장합니다.
# 채운님이 CB 모델에, 선우님이 시각화/매핑에 사용
meta_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl')
# .to_pickle()로 저장
all_meta_df.to_pickle(meta_output_path)
print(f"메타 데이터 저장 완료: {meta_output_path} ({len(all_meta_df)} 행)")

print("\n--- (1) 모든 전처리 및 파일 저장 완료 ---")

리뷰 데이터 저장 완료: ../output\preprocessed_reviews.csv (1521519 행)
메타 데이터 저장 완료: ../output\preprocessed_metadata.pkl (86889 행)

--- (1) 모든 전처리 및 파일 저장 완료 ---


In [16]:
############################################################
# 2. 임베딩 (Embedding) - TF-IDF
############################################################

# 채운님이 사용할 수 있도록 전처리된 텍스트('cleaned_content')를 숫자 벡터(TF-IDF)로 변환(임베딩)


from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import os
import joblib 
from scipy.sparse import save_npz 

print("TF-IDF 및 모델 저장 라이브러리 임포트 완료.")

TF-IDF 및 모델 저장 라이브러리 임포트 완료.


In [17]:
# [중요] Cell 11에서 저장한 '메타 데이터'를 다시 불러옵니다.

# 저장했던 pkl 파일 경로
meta_path = os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl')
# pd.read_pickle()로 pkl 파일을 읽어옴 (데이터 구조가 완벽히 복원됨)
meta_df = pd.read_pickle(meta_path)

# 로드된 데이터 정보 확인
print(f"로드된 메타 데이터: {meta_df.shape}")
# TF-IDF의 입력이 될 'cleaned_content' 컬럼 확인
print(meta_df[['asin', 'title_str', 'cleaned_content']].head())

로드된 메타 데이터: (86889, 21)
              asin                                   title_str  \
213593  B000002X4C                        Release Some Tension   
213607  B000G7PNI6                                  Elvis Live   
213610  B008K8G530  Jekyll & Hyde 2012 Concept Recording / Var   
213631  B000OONQAI  Celebration of Gospel - Taking You Higher!   
213633  B00009EPYE        Yellowstone for Violin and Orchestra   

                                          cleaned_content  
213593      release some tension swv release some tension  
213607                 elvis live presleyelvis elvis live  
213610  jekyll hyde concept recording var concept reco...  
213631  celebration of gospel taking you higher produc...  
213633  yellowstone for violin and orchestra product d...  


In [18]:
# TfidfVectorizer 객체를 생성하고, 텍스트 데이터에 학습시킵니다.

# 1. 벡터라이저 초기화 (하이퍼파라미터 설정)
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',  # 영어 불용어(a, the, is...) 자동 제거
    min_df=5,             # 5개 미만의 문서(아이템)에만 나타난 너무 희귀한 단어는 무시
    max_df=0.7,           # 70% 이상의 문서(아이템)에 나타난 너무 흔한 단어는 무시
    max_features=20000    # 최대 20,000개의 주요 단어만 피처로 사용 (메모리 관리)
)

print("TF-IDF 벡터라이저 초기화 완료. 'cleaned_content' 학습 시작...")

# 2. .fit_transform() 호출:
#    (1) .fit(): 'cleaned_content'를 읽고 20,000개의 단어 사전을 구축 (학습)
#    (2) .transform(): 'cleaned_content'의 각 텍스트를 20,000개 단어의 TF-IDF '숫자 벡터'로 변환
tfidf_matrix = tfidf_vectorizer.fit_transform(meta_df['cleaned_content'])

# tfidf_matrix는 (아이템 수 x 20,000) 형태의 '희소 행렬(Sparse Matrix)'입니다.
# (희소 행렬: 대부분 0인 행렬을 효율적으로 저장하는 방식)

print("TF-IDF 학습 및 변환 완료.")
print(f"TF-IDF Matrix 형태: {tfidf_matrix.shape}")
print(f"(총 {tfidf_matrix.shape[0]}개의 아이템이 각각 {tfidf_matrix.shape[1]}개의 단어 벡터로 표현됨)")

TF-IDF 벡터라이저 초기화 완료. 'cleaned_content' 학습 시작...
TF-IDF 학습 및 변환 완료.
TF-IDF Matrix 형태: (86889, 20000)
(총 86889개의 아이템이 각각 20000개의 단어 벡터로 표현됨)


In [19]:
# [중요]  2개의 결과물을 저장
# 1. tfidf_matrix: 실제 숫자 벡터가 담긴 행렬 (.npz)
# 2. tfidf_vectorizer: 이 행렬을 만든 '모델(변환기)' 자체 (.joblib)

# 저장 경로 설정
matrix_path = os.path.join(OUTPUT_DIR, 'tfidf_embedding_matrix.npz')
vectorizer_path = os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.joblib')

# 1. TF-IDF 행렬(희소 행렬) 저장
#    (희소 행렬은 .to_csv()로 저장이 안되므로, save_npz를 사용)
save_npz(matrix_path, tfidf_matrix)
print(f"TF-IDF 행렬 저장 완료: {matrix_path}")

# 2. TF-IDF 벡터라이저(모델) 저장
#    (모델 객체는 joblib.dump를 사용해 저장)
joblib.dump(tfidf_vectorizer, vectorizer_path)
print(f"TF-IDF 벡터라이저 모델 저장 완료: {vectorizer_path}")

print("\n--- (2) 1번 역할의 '임베딩' 단계 산출물 저장 완료 ---")

# 방금 저장한 `output/` 폴더의 파일 2개(+1개)를 사용하여 **'코사인 유사도'**를 계산할 수 있습니다.

TF-IDF 행렬 저장 완료: ../output\tfidf_embedding_matrix.npz
TF-IDF 벡터라이저 모델 저장 완료: ../output\tfidf_vectorizer.joblib

--- (2) 1번 역할의 '임베딩' 단계 산출물 저장 완료 ---


In [20]:
############################################################
# 3. 트렌드 (Popularity-Based) 모델
############################################################

# Cold-Start 사용자를 위한 '인기 아이템' (Top-N) 추천 모델을 구축합니다.

import pandas as pd
import numpy as np
import os

print("\n--- 3. 트렌드 모델 구축 시작 ---")

# 1. 파일 로드 (리뷰, 메타)
# (이미 저장된 전처리 완료 데이터를 사용)
reviews_path = os.path.join(OUTPUT_DIR, 'preprocessed_reviews.csv')
meta_path = os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl')

reviews_df = pd.read_csv(reviews_path)
meta_df = pd.read_pickle(meta_path)


--- 3. 트렌드 모델 구축 시작 ---


In [21]:
# '인기'를 정의하기 위해, 'asin' (아이템) 기준으로 그룹화
item_stats = reviews_df.groupby('asin').agg(
    avg_rating=('overall', 'mean'),  # 'overall' 컬럼의 '평균'을 'avg_rating'으로
    review_count=('overall', 'size') # 'overall' 컬럼의 '개수'를 'review_count'로
).reset_index() # 그룹화된 'asin'을 다시 컬럼으로 변환

print("아이템별 통계 계산 완료:")
print(item_stats.head())

# (참고) 전처리(Cell 8)에서 N=10으로 필터링했기 때문에,
# 모든 아이템은 최소 10개 이상의 리뷰를 가지고 있습니다.
print(f"\n최소 리뷰 수 (m의 기준): {item_stats['review_count'].min()}")

아이템별 통계 계산 완료:
         asin  avg_rating  review_count
0  0001393774    5.000000             5
1  0005164885    4.767442            43
2  0739045059    4.750000             8
3  0740316826    5.000000             2
4  0767804341    4.733333            15

최소 리뷰 수 (m의 기준): 1


In [22]:
# [문제] 평점 5.0 (리뷰 10개) vs 평점 4.8 (리뷰 1000개) -> 후자가 더 신뢰도 높음
# [해결] IMDb의 가중 평점(Weighted Rating) 공식을 사용해 '신뢰도'를 보정
#
# W = (v / (v+m)) * R + (m / (v+m)) * C
#
# v = 아이템별 리뷰 수 (item_stats['review_count'])
# m = 최소 투표 수 (신뢰도를 얻기 위한 최소 리뷰 수. Cell 8의 N=10을 사용)
# R = 아이템별 평균 평점 (item_stats['avg_rating'])
# C = 전체 아이템의 평균 평점 (전체 평균)

# 1. C: 전체 평균 평점 계산
C = reviews_df['overall'].mean()
print(f"전체 평균 평점 (C): {C:.4f}")

# 2. m: 최소 투표 수 (Cell 8에서 10개 미만은 이미 필터링했으므로 10 사용)
m = 10 
print(f"최소 리뷰 수 기준 (m): {m}")

# 3. 가중 평점(score) 계산 함수
def calculate_weighted_rating(row):
    # 각 행(row)에서 v와 R을 가져옴
    v = row['review_count']
    R = row['avg_rating']
    # m과 C는 위에서 계산한 전역 변수 사용
    score = (v / (v + m)) * R + (m / (v + m)) * C
    return score

# .apply(axis=1)를 사용해 모든 행(row)에 대해 함수 적용
# 'score' 컬럼에 계산된 가중 평점 저장
item_stats['score'] = item_stats.apply(calculate_weighted_rating, axis=1)

print("가중 평점(score) 계산 완료.")
print(item_stats.sort_values('score', ascending=False).head()) # 점수 높은 순으로 정렬해 확인

전체 평균 평점 (C): 4.4872
최소 리뷰 수 기준 (m): 10
가중 평점(score) 계산 완료.
             asin  avg_rating  review_count     score
2419   B000000XDJ    4.979592            98  4.934000
26749  B00005RIKI    4.955556           135  4.923256
6905   B000002H38    4.939655           116  4.903746
75833  B01843CZZC    4.926829           164  4.901564
62371  B007V4QLH4    4.957746            71  4.899655


In [23]:
# 1. 계산된 'score'를 기준으로 내림차순 정렬
popular_items_df = item_stats.sort_values('score', ascending=False)

# 2. 메타데이터(title, domain)와 조인(merge)
#    (결과가 'asin'만 있으면 알아보기 힘드므로, 'title'과 'domain' 정보를 합침)
final_popular_df = popular_items_df.merge(
    meta_df[['asin', 'title_str', 'domain']], # 메타에서 필요한 컬럼만 선택
    on='asin',       # 'asin'을 기준으로
    how='left'       # item_stats(left) 기준으로 합침
)

# 3. 최종 Top-N (e.g., Top 20) 확인
TOP_N = 20
print(f"\n--- 🏆 전체 Top {TOP_N} 인기 아이템 (트렌드 모델) ---")
# 보기 좋게 필요한 컬럼만 선택하여 출력
print(final_popular_df[['asin', 'title_str', 'domain', 'score', 'avg_rating', 'review_count']].head(TOP_N))

# 4. (선택적) 도메인별 Top-N
# 'instruments' 도메인 Top 10
pop_inst = final_popular_df[final_popular_df['domain'] == 'instruments'].head(10)
print("\n--- Top 10 인기 악기 ---")
# .to_markdown()은 Jupyter Notebook에서 표(table)로 깔끔하게 출력해줌
print(pop_inst[['title_str', 'score']].to_markdown(index=False))

# 'music' 도메인 Top 10
pop_music = final_popular_df[final_popular_df['domain'] == 'music'].head(10)
print("\n--- Top 10 인기 음반 ---")
print(pop_music[['title_str', 'score']].to_markdown(index=False))

# 5. 이 '트렌드 모델' 자체(DataFrame)를 CSV로 저장
# (4번(선우) 역할이 Cold Start 사용자에게 이 리스트를 바로 보여줄 수 있도록)
popular_model_path = os.path.join(OUTPUT_DIR, 'popularity_model_top_items.csv')
final_popular_df.to_csv(popular_model_path, index=False, encoding='utf-8')

print(f"\n--- (3) '트렌드 모델' 결과 저장 완료: {popular_model_path} ---")
print("\n==============================================")
print("모든 작업(전처리, 임베딩, 트렌드) 완료.")
print("==============================================")


--- 🏆 전체 Top 20 인기 아이템 (트렌드 모델) ---
          asin                                          title_str domain  \
0   B000000XDJ  A Charlie Brown Christmas: The Original Sound ...  music   
1   B00005RIKI                  Pronounced 'Lĕh-'nérd 'Skin-'nérd  music   
2   B000002H38                                            Rapture  music   
3   B01843CZZC                     Hymns That Are Important To Us  music   
4   B007V4QLH4                                          Unplugged  music   
5   B074JHP7QY                                      Inspirational  music   
6   B00T4LLMO4                                               Road  music   
7   B015C5MFSC                                           The Hill  music   
8   B0000024IE       Fear Of A Black Planet       explicit_lyrics  music   
9   B08TQHTBYG                                     Believe For It  music   
10  B0076FJ2R4                                  Precious Memories  music   
11  B00MGSTO4C                                Cd Fr